# TP1 — Lab 1: Regular Expressions

A worked Jupyter notebook for the eight exercises, discussion questions, and information-extraction mini-challenge in Lab 1. It brings together the material in exo1.py through exo8.py and lab1.py. The original handout is [Lab-1.pdf](Lab-1.pdf).

Run code cells from top to bottom. The examples use Python's built-in re module; no extra package is needed for TP1.

In [1]:
import re

## Exercise 1 — Literal matching and counting

Start with the three-sentence corpus from the handout. Practise literal matching, whole-word matching, alternation, case-insensitive matching, and counting sentences that mention either topic.

In [2]:
text = "NLP is useful. NLP is interesting. Machine learning is useful."

print("NLP:", re.findall(r"NLP", text))
print("useful:", re.findall(r"useful", text))
print("whole-word NLP:", re.findall(r"\bNLP\b", text + " NLPs"))


def count_occurrences(pattern, text, case_sensitive=True):
    flags = 0 if case_sensitive else re.IGNORECASE
    return len(re.findall(pattern, text, flags=flags))

print("useful OR interesting:", count_occurrences(r"useful|interesting", text))
mixed_case = "NLP is useful. nlp is interesting. Nlp is useful. NLPs are different."
print("case-insensitive whole-word NLP:", count_occurrences(r"\bNLP\b", mixed_case, case_sensitive=False))

sentences = [part.strip() for part in text.split(".") if part.strip()]
topic_pattern = re.compile(r"\b(?:NLP|machine learning)\b", re.IGNORECASE)
hits = [sentence for sentence in sentences if topic_pattern.search(sentence)]
print(f"Topic sentences: {len(hits)}/{len(sentences)} ({100 * len(hits) / len(sentences):.0f}%)")

NLP: ['NLP', 'NLP']
useful: ['useful', 'useful']
whole-word NLP: ['NLP', 'NLP']
useful OR interesting: 3
case-insensitive whole-word NLP: 3
Topic sentences: 3/3 (100%)


**Questions and observations**

- NLP occurs twice; useful occurs twice. The alternation useful|interesting finds either term, for three matches total.
- The boundary pattern \bNLP\b matches NLP as a complete word and rejects NLPs. Case-insensitive matching also accepts nlp and Nlp.
- The three corpus sentences all mention NLP or machine learning, so the challenge result is 100%.
- A literal NLP search looks for one exact string. Alternation searches for either of two alternatives. A word boundary prevents a match inside a longer word.

## Exercise 2 — Character classes and a username validator

Find words ending in at, identify three-letter consonant-only words, and validate the four supplied usernames. The length requirement is 4–12 characters, with lowercase letters, digits, and underscore only.

In [3]:
text = "cat bat rat dog hot"
print("Words ending in at:", re.findall(r"\b\w*at\b", text))

consonant_text = "cat sky crypt shy"
print("Three-letter consonant words:", re.findall(r"\b[bcdfghjklmnpqrstvwxyz]{3}\b", consonant_text))

usernames = ["ali_92", "A", "student.name", "nlp_lab_2026"]
username_pattern = re.compile(r"[a-z0-9_]{4,12}")

for username in usernames:
    valid = username_pattern.fullmatch(username) is not None
    if valid:
        reason = ""
    else:
        reasons = []
        if len(username) < 4:
            reasons.append("too short (minimum 4 characters)")
        if len(username) > 12:
            reasons.append("too long (maximum 12 characters)")
        if re.search(r"[^a-z0-9_]", username):
            reasons.append("contains characters outside lowercase letters, digits, and underscore")
        reason = "; ".join(reasons)
    print(f"{username}: {'VALID' if valid else 'INVALID'}{(' — ' + reason) if reason else ''}")

Words ending in at: ['cat', 'bat', 'rat']
Three-letter consonant words: ['sky', 'shy']
ali_92: VALID
A: INVALID — too short (minimum 4 characters); contains characters outside lowercase letters, digits, and underscore
student.name: INVALID — contains characters outside lowercase letters, digits, and underscore
nlp_lab_2026: VALID


**Questions and observations**

- [cb] means one character, either c or b, so [cb]at matches cat and bat. A character class describes allowed characters; the {4,12} quantifier sets the allowed length.
- ali_92 and nlp_lab_2026 are valid. A is too short and uppercase; student.name contains a period, which is forbidden.
- A consonant character class alone does not impose a word boundary or a length. The full pattern uses both.

## Exercise 3 — Digits, decimals, and prices

Extract ordinary numbers and signed dollar amounts from the receipt. Convert the monetary values to floats, compute their sum, and separate non-currency quantities from prices.

In [4]:
receipt = "2 apples, 1.5 kg of rice, 3 juices at $2.50 each, total: $12.75 (-$1.00 discount)."
number_pattern = r"-?\d+(?:\.\d+)?"
money_pattern = r"-?\$\d+(?:\.\d+)?"

print("Numbers:", re.findall(number_pattern, receipt))
money_strings = re.findall(money_pattern, receipt)
amounts = [float(value.replace("$", "")) for value in money_strings]
print("Dollar amounts:", money_strings)
print("Amounts as floats:", amounts)
print("Sum:", sum(amounts))

# Remove the monetary amounts first, then extract what remains.
non_currency_text = re.sub(money_pattern, "", receipt)
quantities = re.findall(r"\b\d+(?:\.\d+)?\b", non_currency_text)
print("Non-currency quantities:", quantities)


Numbers: ['2', '1.5', '3', '2.50', '12.75', '1.00']
Dollar amounts: ['$2.50', '$12.75', '-$1.00']
Amounts as floats: [2.5, 12.75, -1.0]
Sum: 14.25
Non-currency quantities: ['2', '1.5', '3']


**Questions and observations**

- \d matches a digit; + means one or more repetitions. Therefore \d+ stops before a decimal point, which is not a digit.
- The optional decimal part (?:\.\d+)? keeps values such as 1.5 together. The optional leading minus sign handles negative amounts. The dollar sign is escaped because it has a special meaning in a regular expression.
- The monetary values are 2.50, 12.75, and -1.00; their sum is 14.25. The non-currency quantities are 2, 1.5, and 3.
- A real receipt may use different currency symbols, separators, or accounting notation, so this pattern is deliberately limited to the exercise format.

## Exercise 4 — Dates and times in mixed formats

Handle DD/MM/YYYY, DD-MM-YYYY, YYYY-MM-DD, and times written with either a colon or h. The schedule includes a one-digit hour.

In [5]:
schedule = (
    "Lab 1: 16/09/2026, Lab 2: 2026-09-23, Lab 3: 30-09-2026, "
    "exam at 9:00, revision session at 14:30, office hours 09h30."
)

local_date_pattern = r"\b\d{2}[/-]\d{2}[/-]\d{4}\b"
iso_date_pattern = r"\b\d{4}-\d{2}-\d{2}\b"
all_dates_pattern = r"\b(?:\d{2}[/-]\d{2}[/-]\d{4}|\d{4}-\d{2}-\d{2})\b"
time_pattern = r"\b\d{1,2}(?::|h)\d{2}\b"

print("DD/MM or DD-MM dates:", re.findall(local_date_pattern, schedule))
print("ISO dates:", re.findall(iso_date_pattern, schedule))
print("All dates:", re.findall(all_dates_pattern, schedule))
print("Times:", re.findall(time_pattern, schedule))

# Regex checks the shape. datetime checks whether the calendar date exists.
from datetime import datetime
for value in re.findall(all_dates_pattern, schedule):
    fmt = "%Y-%m-%d" if value[4] == "-" and value[:4].isdigit() else ("%d/%m/%Y" if "/" in value else "%d-%m-%Y")
    datetime.strptime(value, fmt)
print("All extracted dates are valid calendar dates.")

DD/MM or DD-MM dates: ['16/09/2026', '30-09-2026']
ISO dates: ['2026-09-23']
All dates: ['16/09/2026', '2026-09-23', '30-09-2026']
Times: ['9:00', '14:30', '09h30']
All extracted dates are valid calendar dates.


**Questions and observations**

- \d{2} means exactly two digits, the slash or hyphen is a literal separator, and \d{4} means four digits. The combined expression adds an alternative for the ISO order.
- The time expression uses \d{1,2} so that 9:00 and 14:30 both match, and (?: : | h ) accepts either separator (without the spaces shown here).
- Fixed-width day and month fields miss 6/9/2026. A more permissive shape is \d{1,2}[/-]\d{1,2}[/-]\d{4}.
- A regex can check the shape but may accept impossible dates such as 31/02/2026. Parse the match with a date library to validate the calendar.

## Exercise 5 — Email addresses and a validity checker

Compare the handout's starter expression with a stricter teaching pattern. Email syntax has many exceptions; the improved pattern covers the supplied examples rather than every address allowed by the mail standards.

In [6]:
mailing_list = [
    "ali@example.com", "student.name@univ.dz", "a.b-c@sub.univ.dz",
    "not-an-email", "missing@domain", "double@@example.com",
    "@no-local-part.com", "ok.name+lab@example.co.uk",
]
original_pattern = re.compile(r"[\w.-]+@[\w.-]+\.\w+")
final_pattern = re.compile(
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}"
)

for address in mailing_list:
    print(f"{address:32} starter finds={bool(original_pattern.search(address))!s:5} ", end="")
    print(f"improved valid={final_pattern.fullmatch(address) is not None}")

valid_addresses = [address for address in mailing_list if final_pattern.fullmatch(address)]
invalid_addresses = [address for address in mailing_list if final_pattern.fullmatch(address) is None]
print(f"Valid ({len(valid_addresses)}): {valid_addresses}")
print(f"Invalid ({len(invalid_addresses)}): {invalid_addresses}")

ali@example.com                  starter finds=True  improved valid=True
student.name@univ.dz             starter finds=True  improved valid=True
a.b-c@sub.univ.dz                starter finds=True  improved valid=True
not-an-email                     starter finds=False improved valid=False
missing@domain                   starter finds=False improved valid=False
double@@example.com              starter finds=False improved valid=False
@no-local-part.com               starter finds=False improved valid=False
ok.name+lab@example.co.uk        starter finds=True  improved valid=True
Valid (4): ['ali@example.com', 'student.name@univ.dz', 'a.b-c@sub.univ.dz', 'ok.name+lab@example.co.uk']
Invalid (4): ['not-an-email', 'missing@domain', 'double@@example.com', '@no-local-part.com']


**Questions and observations**

- The starter expression has a local part before @, a domain after it, and a dot plus a final label. It can still accept malformed input because its domain character class includes @ and the search need not consume the whole string.
- The improved expression uses fullmatch, requires a nonempty local part, allows + in that part, and requires a dotted domain with a two-letter-or-longer alphabetic final label. For the given list, ali@example.com, student.name@univ.dz, a.b-c@sub.univ.dz, and ok.name+lab@example.co.uk pass.
- This teaching pattern can reject valid edge cases, including addresses with a one-letter top-level domain or a domain with no dot such as user@localhost. Full email validity depends on standards and domain rules and is better confirmed with a validation library or a verification message.

## Exercise 6 — URLs and domain extraction

Extract web links while excluding punctuation around them, extract each web link's host, and separately demonstrate how FTP can be included.

In [7]:
links_text = (
    "See https://www.univ.dz/lab1?page=2, http://example.org, "
    "https://docs.python.org/3/library/re.html and "
    "ftp://old-server.com (not a web link)."
)
naive_url_pattern = r"https?://[^\s]+"
print("Naive extraction (punctuation may remain):", re.findall(naive_url_pattern, links_text))
web_url_pattern = r"https?://[^\s,()<>]*[^\s,().,<>]"
all_protocols_pattern = r"(?:https?|ftp)://[^\s,()<>]*[^\s,().,<>]"
web_urls = re.findall(web_url_pattern, links_text)
all_urls = re.findall(all_protocols_pattern, links_text)
domain_pattern = re.compile(r"^https?://([^/:?#]+)")
domains = [domain_pattern.match(url).group(1) for url in web_urls]

print("HTTP/HTTPS links:", web_urls)
print("Domains:", domains)
print("Including FTP:", all_urls)
print("HTTPS count:", sum(url.startswith("https://") for url in web_urls))
print("HTTP count:", sum(url.startswith("http://") for url in web_urls))

Naive extraction (punctuation may remain): ['https://www.univ.dz/lab1?page=2,', 'http://example.org,', 'https://docs.python.org/3/library/re.html']
HTTP/HTTPS links: ['https://www.univ.dz/lab1?page=2', 'http://example.org', 'https://docs.python.org/3/library/re.html']
Domains: ['www.univ.dz', 'example.org', 'docs.python.org']
Including FTP: ['https://www.univ.dz/lab1?page=2', 'http://example.org', 'https://docs.python.org/3/library/re.html', 'ftp://old-server.com']
HTTPS count: 2
HTTP count: 1


**Questions and observations**

- https? means http followed by an optional s. [^\s]+ means one or more characters that are not whitespace; this simple starter can swallow punctuation.
- The extraction pattern stops at common surrounding punctuation. It finds two HTTPS links and one HTTP link; FTP is ignored until ftp is added as an alternative.
- URL syntax permits many punctuation characters, so punctuation stripping is ambiguous: a comma or closing parenthesis can be part of a URL in some contexts. Production URL parsing should use a parser and context-aware cleanup.

## Exercise 7 — Substitution and a mini PII-redaction tool

Replace email addresses, phone numbers, and dates, while leaving ordinary course information such as 20-hour unchanged. The phone expression accepts common Algerian national and international forms used in the handout.

In [8]:
number_example = "The course has 20 hours and 12 exercises."
print(re.sub(r"\d+", "<NUMBER>", number_example))

comment = (
    "Contact me at ali.ben@example.com or call 0555 12 34 56 "
    "if you have questions about the 20-hour course. — Ali, 16/09/2026."
)
email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}"
phone_pattern = r"(?:\+213(?:[\s-]?\d){9}|0(?:[\s-]?\d){9})"
date_pattern = r"\b\d{2}[/-]\d{2}[/-]\d{4}\b"

def anonymize(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)
    text = re.sub(date_pattern, "<DATE>", text)
    return text

print(anonymize(comment))

phone_examples = ["0555123456", "0555 12 34 56", "+213 555 12 34 56"]
for phone in phone_examples:
    print(phone, "->", re.sub(phone_pattern, "<PHONE>", phone))

The course has <NUMBER> hours and <NUMBER> exercises.
Contact me at <EMAIL> or call <PHONE> if you have questions about the 20-hour course. — Ali, <DATE>.
0555123456 -> <PHONE>
0555 12 34 56 -> <PHONE>
+213 555 12 34 56 -> <PHONE>


**Questions and observations**

- re.sub is useful when each match should be replaced with a label. Email addresses, phones, and dates are replaced; the number 20 stays because it is not covered by any of those patterns.
- Phone numbers are difficult because people vary their grouping, spaces, and international prefixes. The pattern here handles the three demonstrated formats but is not a universal phone-number validator.
- Redaction can help prepare public feedback or logs, but pattern matching may miss unusual formats or redact non-personal text. Review is important when complete de-identification matters.

## Exercise 8 — Regex analysis and limitations

Apply the three patterns from the handout to one sentence. Inspect what each expression treats as a number, capitalized word, or word-like sequence.

In [9]:
sentence = "Dr. O'Brien met 3 students in New-York on 16/09/2026."
patterns = {
    r"\d+": r"\d+",
    r"[A-Z][a-z]+": r"[A-Z][a-z]+",
    r"\b\w+\b": r"\b\w+\b",
}
for label, pattern in patterns.items():
    print(f"{label:16} -> {re.findall(pattern, sentence)}")

\d+              -> ['3', '16', '09', '2026']
[A-Z][a-z]+      -> ['Dr', 'Brien', 'New', 'York']
\b\w+\b          -> ['Dr', 'O', 'Brien', 'met', '3', 'students', 'in', 'New', 'York', 'on', '16', '09', '2026']


**Questions and observations**

- `\d+` matches one or more digits, for example `2026` and `42`. It fails on decimals because the dot is not a digit, and it splits dates such as `16/09/2026` into three numbers.
- `[A-Z][a-z]+` matches a capital letter followed by one or more lowercase letters, for example `Nlp` and `Python`. It fails on acronyms such as `NLP`, on a lone initial such as `O`, and on names written in all caps.
- `\b\w+\b` matches a run of word characters between word boundaries, for example `token` and `2026`. It fails on hyphenated and contracted forms: `New-York` and `O'Brien` are split at the punctuation.
- On the test sentence, `\d+` returns `3`, `16`, `09`, and `2026`; `[A-Z][a-z]+` returns `Dr`, `Brien`, `New`, and `York`; `\b\w+\b` returns every word-like chunk, splitting `O'Brien` and `New-York` and breaking the date into three pieces.
- Contractions and apostrophes, hyphenation, acronyms, multi-word names, and multi-part dates all make regex a fragile general-purpose tokenizer. This motivates the dedicated tokenization tools in Lab 2.


## Lab 1 mini-challenge — Information-extraction report

Combine the patterns above on the announcement from lab1.py. The numbers field intentionally returns numeric pieces wherever they occur, including components of dates, times, and phone numbers; the other fields provide structured extractions.

In [10]:
announcement = (
    "Reminder: the NLP project (worth 12 points) is due on 30/09/2026 before 23:59. "
    "Send your report (max 2.5 MB) to nlp.lab@univ.dz or upload it at "
    "https://courses.univ.dz/submit. Late submissions lose -1.5 points per day. "
    "Contact +213 555 98 76 54 for questions."
)

report_patterns = {
    "dates": r"\b\d{2}[/-]\d{2}[/-]\d{4}\b",
    "times": r"\b\d{1,2}(?::|h)\d{2}\b",
    "emails": email_pattern,
    "urls": r"https?://[^\s,()<>]*[^\s,().,<>]",
    "phones": phone_pattern,
    "numbers": r"(?<![\w.])[+-]?\d+(?:\.\d+)?(?![\w.])",
}
report = {name: re.findall(pattern, announcement) for name, pattern in report_patterns.items()}
for name, values in report.items():
    print(f"{name}: {values}")

dates: ['30/09/2026']
times: ['23:59']
emails: ['nlp.lab@univ.dz']
urls: ['https://courses.univ.dz/submit']
phones: ['+213 555 98 76 54']
numbers: ['12', '30', '09', '2026', '23', '2.5', '-1.5', '+213', '555', '98', '76', '54']


**Discussion**

Email is relatively distinctive here because the at sign and dotted domain provide clear structure. Broad numeric extraction is the hardest category: digits also occur inside dates, times, phone numbers, decimal values, and negative values. The numeric output is therefore a list of matches, not a semantic classification of every number.